# 01 — How the experiment was run

This notebook **documents** the run pipeline (it needs an `ANTHROPIC_API_KEY` with
credits to actually execute, so treat it as a recipe; the outputs it produced are
frozen in `results/` and analyzed — without any API key — in `02_analysis.ipynb`).

**Design**: see `docs/design.md` (rulebook v1.2, every decision pre-registered in its changelog).

The pipeline, in order:

| step | command | output (frozen) |
|---|---|---|
| 1. build instances + batch request files | `python src/prep_instances.py` | `data/instances_*.csv` |
| 2. pilot (10 Q × Sonnet, audited) | `python src/run_pilot.py` | `results/pilot.csv` |
| 3. M2 paraphrases | `python src/run_paraphrases.py` | `data/paraphrases.csv` |
| 4. rebuild with paraphrase requests | `python src/prep_instances.py --with-para` | — |
| 5. submit batches (Message Batches API) | `python src/run_batches.py submit fb_all fb_para fsq_haiku fsq_sonnet fsq_opus` | `results/batch_ids.json` |
| 6. wait + download | `python src/run_batches.py poll` / `collect` | `results/batch_raw/` |
| 7. freeze raw outputs (sacred) | `python src/freeze_raw.py` | `results/raw_answers.csv` |
| 8. grade (Python numeric + pinned AI grader) | `python src/grade_all.py submit` / `finalize` | `results/grades.csv`, `results/m2_agreement.csv` |

Repo: https://github.com/arshhn/honest-finance-qa


## The ask protocol (verbatim)

Every model sees exactly this (from `src/prompts.py`, pinned):

```
You are answering a question about a company's SEC filing. Use ONLY the provided document text.
Document: {evidence}
Question: {question}
Rules: If the document does not contain enough information, reply exactly "CANNOT ANSWER".
Reply in this format —
ANSWER: <your answer, with units>
CONFIDENCE: <integer 0–100, how likely your answer is correct>
```

Models (exact snapshots recorded per row in `raw_answers.csv`):
small = `claude-haiku-4-5`, mid = `claude-sonnet-5`, large = `claude-opus-5`,
each at its API defaults (no temperature/thinking/effort overrides — see changelog v1.2b).
